<h1>Task 1<h1> 

### Problem Definition ###

- Problem statement: This project develops a multi-model intelligence system to predict the daily probability of wildfire risk in Azerbaijan's forested regions. By moving beyond standard 7-day weather forecasts, we evaluate environmental "risk probability" by synthesizing historical meteorological patterns, high-resolution soil health data, and atmospheric dryness indicators.


- Why it matters: 
    - This project develops a multi-model intelligence system to predict the daily probability of wildfire risk in Azerbaijan's forested regions. By moving beyond standard 7-day weather forecasts, we evaluate environmental "risk probability" by synthesizing historical meteorological patterns, high-resolution soil health data, and atmospheric dryness indicators.
    - Agricultural Planning: Farmers in major hubs like Lankaran and Guba can utilize our secondary soil temperature and moisture forecasts to optimize irrigation cycles and protect crops from seasonal heatwave anomalies.


- Prediction target: 
    - **Primary**: Daily wildfire risk probability (binary classification: 1 = high risk, 0 = low risk)
    - **Secondary**: Daily soil temperature/moisture (Regression) and Summer "Heat Wave" probability (Statistical).


- Cities:
    - 15 regional hubs including Baku, Ganja, Lankaran, Guba, Zaqatala, and Shamakhi. 


- Weather/Soil variables: 
    - temperature_2m_max  
    - temperature_2m_min  
    - precipitation_sum  
    - wind_speed_10m_max  
    - vapour_pressure_deficit  
    - soil_temperature_0_to_7cm  
    - soil_moisture_7_to_28cm

- Other Datas:
    - NASA Satellite images to check actual cloud locations as well as fire occurances, and to detect fuel materials for fire occuring
    - Map of Azerbaijan Forestry(border lines)

<h1>Task 2<h1>

In [ ]:
import openmeteo_requests
import requests_cache
import pandas as pd
import seaborn as sns
from retry_requests import retry
import time 
import numpy as np
import requests
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, r2_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

In [ ]:
LAT, LON = 40.4093, 49.8671

# =========================
# 1. HISTORICAL DATA
# =========================

hist_url = "https://archive-api.open-meteo.com/v1/archive"

hist_params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": "2023-01-01",
    "end_date": "2026-04-19",       #today's date
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max",
    "timezone": "auto"
}

hist_response = requests.get(hist_url, params=hist_params).json()

# DEBUG STEP (VERY IMPORTANT)
if "daily" not in hist_response:
    print("HIST ERROR RESPONSE:")
    print(hist_response)
    raise ValueError("Historical API failed — fix request above")

df_hist = pd.DataFrame(hist_response["daily"])
df_hist["time"] = pd.to_datetime(df_hist["time"])


# =========================
# 2. FORECAST DATA
# =========================

fore_url = "https://api.open-meteo.com/v1/forecast"

fore_params = {
    "latitude": LAT,
    "longitude": LON,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max",
    "timezone": "auto"
}

fore_response = requests.get(fore_url, params=fore_params).json()

if "daily" not in fore_response:
    print("FORECAST ERROR RESPONSE:")
    print(fore_response)
    raise ValueError("Forecast API failed — fix request above")

df_fore = pd.DataFrame(fore_response["daily"])
df_fore["time"] = pd.to_datetime(df_fore["time"])

In [ ]:
 # 1. Setup the API client 
 
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)
url = "https://archive-api.open-meteo.com/v1/archive"



In [ ]:
 # 2. Define your locations (Major districts for country-wide coverage)

azerbaijan_cities = {
    "Baku": {"lat": 40.4093, "lon": 49.8671},
    "Ganja": {"lat": 40.6828, "lon": 46.3606},
    "Lankaran": {"lat": 38.7529, "lon": 48.8515},
    "Guba": {"lat": 41.3597, "lon": 48.5134},
    "Zaqatala": {"lat": 41.6336, "lon": 46.6433},
    "Nakhchivan": {"lat": 39.2089, "lon": 45.4122},
    "Sheki": {"lat": 41.1919, "lon": 47.1706},
    "Shirvan": {"lat": 39.9317, "lon": 48.929},
    "Mingachevir": {"lat": 40.7639, "lon": 47.0595},
    "Khachmaz": {"lat": 41.4635, "lon": 48.806},
    "Goychay": {"lat": 40.6533, "lon": 47.7401},
    "Shamkir": {"lat": 40.8298, "lon": 46.0162},
    "Sabirabad": {"lat": 40.0101, "lon": 48.4772},
    "Imishli": {"lat": 39.8694, "lon": 48.06},
    "Shamakhi": {"lat": 40.6303, "lon": 48.6414}
}

In [ ]:
daily_data = []
hourly_data = []

print(f"Starting pipeline for {len(azerbaijan_cities)} locations...")

# 3. Loop through each city safely with the Smart Retry

for city_name, coords in azerbaijan_cities.items():
    print(f"\nFetching 2 years of Hourly & Daily data for: {city_name}...")

    params = {
        "latitude": coords["lat"],
        "longitude": coords["lon"],
        "start_date": "2024-03-31",
        "end_date": "2026-04-19", # Today's date
        "daily": [
            "temperature_2m_max", "temperature_2m_mean", "apparent_temperature_mean", 
            "temperature_2m_min", "wind_speed_10m_max", "wind_gusts_10m_max", "precipitation_sum"
        ],
        "hourly": [
            "temperature_2m", "soil_temperature_0_to_7cm", "soil_temperature_7_to_28cm", 
            "soil_temperature_28_to_100cm", "soil_temperature_100_to_255cm", "wind_gusts_10m", 
            "wind_speed_10m", "wind_direction_10m", "wind_direction_100m", "wind_speed_100m", 
            "soil_moisture_100_to_255cm", "soil_moisture_28_to_100cm", "soil_moisture_7_to_28cm", 
            "soil_moisture_0_to_7cm", "relative_humidity_2m", "precipitation", "vapour_pressure_deficit"
        ]
    }
    success = False
    
    while not success:
        try:
            responses = openmeteo.weather_api(url, params=params)
            response = responses[0]
            
            # --- PROCESS DAILY DATA ---
            daily = response.Daily()
            city_daily_df = pd.DataFrame({
                "date": pd.date_range(
                    start=pd.to_datetime(daily.Time(), unit="s", utc=True),
                    end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
                    freq=pd.Timedelta(seconds=daily.Interval()),
                    inclusive="left"
                ),
                "location": city_name,
                "latitude": coords["lat"],
                "longitude": coords["lon"],
                "temperature_2m_max": daily.Variables(0).ValuesAsNumpy(),
                "temperature_2m_mean": daily.Variables(1).ValuesAsNumpy(),
                "apparent_temperature_mean": daily
.Variables(2).ValuesAsNumpy(),
                "temperature_2m_min": daily.Variables(3).ValuesAsNumpy(),
                "wind_speed_10m_max": daily.Variables(4).ValuesAsNumpy(),
                "wind_gusts_10m_max": daily.Variables(5).ValuesAsNumpy(),
                "precipitation_sum": daily.Variables(6).ValuesAsNumpy()
            })
            daily_data.append(city_daily_df)
            
            # --- PROCESS HOURLY DATA ---
            hourly = response.Hourly()
            city_hourly_df = pd.DataFrame({
                "date": pd.date_range(
                    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                    freq=pd.Timedelta(seconds=hourly.Interval()),
                    inclusive="left"
                ),
                "location": city_name,
                "latitude": coords["lat"],
                "longitude": coords["lon"],
                "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
                "soil_temperature_0_to_7cm": hourly.Variables(1).ValuesAsNumpy(),
                "soil_temperature_7_to_28cm": hourly.Variables(2).ValuesAsNumpy(),
                "soil_temperature_28_to_100cm": hourly.Variables(3).ValuesAsNumpy(),
                "soil_temperature_100_to_255cm": hourly.Variables(4).ValuesAsNumpy(),
                "wind_gusts_10m": hourly.Variables(5).ValuesAsNumpy(),
                "wind_speed_10m": hourly.Variables(6).ValuesAsNumpy(),
                "wind_direction_10m": hourly.Variables(7).ValuesAsNumpy(),
                "wind_direction_100m": hourly.Variables(8).ValuesAsNumpy(),
                "wind_speed_100m": hourly.Variables(9).ValuesAsNumpy(),
                "soil_moisture_7_to_28cm": hourly.Variables(12).ValuesAsNumpy(),
                "soil_moisture_0_to_7cm": hourly.Variables(13).ValuesAsNumpy(),
                "relative_humidity_2m": hourly.Variables(14).ValuesAsNumpy(),
                "precipitation": hourly.Variables(15).ValuesAsNumpy(),
                "vapour_pressure_deficit": hourly.Variables(16).ValuesAsNumpy()
            })
            hourly_data.append(city_hourly_df)
            
            print(f" -> Success! Data retrieved for {city_name}.")
            success = True # Breaks the while loop so it moves to the next city
            
            # A healthy 5-second pause between successful pulls
            time.sleep(5) 
            
        except Exception as e:
            error_message = str(e)
            if 'Minutely API request limit' in error_message:
                print(" -> Rate limit hit. Server needs a break. Waiting 60 seconds...")
                time.sleep(60) # Wait a full minute
            else:
                print(f" -> Critical error for {city_name}: {e}")
                break # If it's a different error, just skip the city entirely


In [ ]:
 # 4. Combine all the data!

print("\nMerging datasets...")
final_daily_df = pd.concat(daily_data, ignore_index=True)
final_hourly_df = pd.concat(hourly_data, ignore_index=True) 

In [ ]:
 # 5. Save to separate CSVs so your teammates can start working

final_daily_df.to_csv("aze_regional_daily_weather.csv", index=False)
final_hourly_df.to_csv("aze_regional_hourly_weather.csv", index=False)
print(f"Pipeline Complete! Saved {len(final_daily_df)} Daily rows and {len(final_hourly_df)} Hourly rows.") 

In [ ]:
layer = gpd.read_file("./forest_borders/azerbaijan.kmz", driver="LIBKML") 
m = folium.Map(
    location=[40.4093, 49.8671],
    zoom_start=10,
    tiles="Esri WorldImagery"
)

folium.GeoJson(layer.to_json(), name="Azerbaijan").add_to(m)
folium.LayerControl().add_to(m)

m

In [ ]:
# Report for Historical Data (Task 2)
print("\n--- Historical Data Summary for Hourly Data ---")
print(f"Shape: {final_hourly_df.shape}")
print(f"Columns: {final_hourly_df.columns.tolist()}")
print(f"Date Range: {final_hourly_df['date'].min()} to {final_hourly_df['date'].max()}")
print(f"Missing Values:\n{final_hourly_df.isnull().sum()[final_hourly_df.isnull().sum() > 0]}" if final_hourly_df.isnull().sum().any() else "No missing values.") # Only show columns with missing values

print("\n") # Just a line break for better readability

print("--- Historical Data Summary for Daily Data ---")
print(f"Shape: {final_daily_df.shape}")
print(f"Columns: {final_daily_df.columns.tolist()}")
print(f"Date Range: {final_daily_df['date'].min()} to {final_daily_df['date'].max()}")
print(f"Missing Values:\n{final_daily_df.isnull().sum()[final_daily_df.isnull().sum() > 0]}" if final_daily_df.isnull().sum().any() else "No missing values.") # Only show columns with missing values

print ("\n") # Just a line break for better readability

# Data for Baku
f_baku_hourly = final_hourly_df[final_hourly_df['location'] == 'Baku']
f_baku_daily = final_daily_df[final_daily_df['location'] == 'Baku']


# Visualisation
fig, ax = plt.subplots(3, 1, figsize=(12, 24))

ax[0].plot(f_baku_daily['date'], f_baku_daily['temperature_2m_max'])
ax[0].set_title("Baku Max Temp (2 Years)")

ax[1].plot(f_baku_daily['date'], f_baku_daily['precipitation_sum'])
ax[1].set_title("Baku Total Precipitation (2 Years)")

ax[2].plot(f_baku_daily['date'], f_baku_daily['wind_speed_10m_max'])
ax[2].set_title("Baku Max Wind Speed (2 Years)")




## Overview

The project uses both historical and forecast meteorological data for Baku, Azerbaijan collected from the Open-Meteo API. The dataset includes daily weather variables over a multi-year period, combined with hourly environmental measurements for higher temporal resolution analysis.

---

## Historical Weather Data (Baku)

The historical dataset contains daily weather observations spanning from **2023-01-01 to 2026-04-19**, resulting in **1205 records**.

The dataset includes the following features:
- Maximum and minimum air temperature (2m above ground)
- Total daily precipitation
- Maximum wind speed at 10 meters

All variables are continuous numerical measurements.

The dataset is fully complete with **no missing values**, indicating high data reliability and successful API retrieval without gaps or corruption.

---

## Forecast Weather Data (Baku)

The forecast dataset includes **7 days of predicted weather conditions**, covering the period from **2026-04-20 to 2026-04-26**.

It contains the same feature structure as the historical dataset:
- temperature_2m_max
- temperature_2m_min
- precipitation_sum
- wind_speed_10m_max

Like the historical dataset, the forecast data has **no missing values**, ensuring consistency between training and prediction inputs.

---

## Hourly Environmental Dataset

The hourly dataset provides high-resolution environmental measurements across multiple regions in Azerbaijan.

- Shape: **270,000 rows × 19 features**
- Time range: **2024-03-31 - 2026-04-19**
- Coverage: multiple cities across Azerbaijan

Key variables include:
- Soil temperature at multiple depths
- Soil moisture layers
- Wind speed and direction (10m and 100m)
- Relative humidity
- Vapour pressure deficit
- Precipitation intensity

The dataset is fully complete with **no missing values**, making it suitable for time-series and machine learning feature engineering.

---

## Daily Multi-City Dataset

The daily dataset aggregates weather conditions across 15+ Azerbaijani cities.

- Shape: **11,250 rows × 11 features**
- Time range: **2024-03-31 - 2026-04-19**

Key features:
- Temperature (max, min, mean)
- Apparent temperature
- Wind speed and gusts
- Precipitation sum
- Geographic coordinates (latitude, longitude)

This dataset is also **fully complete with no missing values**, confirming consistent API extraction across all regions.


---

## Visualization Analysis (Baku Case Study)

Our time-series visualizations reveal the critical environmental stressors necessary for wildfire prediction:

Key features:
- Temperature (Seasonal Bell Curve): Shows sharp peaks consistently hitting 35–40°C during summer months. This thermal inertia is a primary driver of the "Ignition Phase" in forest fuels.

- Precipitation (Volatility & Drought): Significant drought windows (zero rainfall) precede summer heat spikes, validating the use of the days_since_rain feature to track fuel desiccation.

- Wind Dynamics: Peak gusts frequently hit 40–50 km/h. In wildfire science, high wind speed acts as a "Force Multiplier," responsible for rapid fire spread and oxygenation.

---

## Data Quality

Overall, the dataset is:
- Complete (0 missing values across all tables)
- Temporally consistent
- Geographically diverse
- Structurally uniform across cities

This makes it highly suitable for machine learning modeling without requiring imputation or data cleaning at the missing-value level.



<h1>Task 3<h1>

In [ ]:

# =========================
# LOAD + SORT DATA
# =========================

df = pd.read_csv("aze_regional_daily_weather.csv")
df = df.sort_values(["location", "date"]).reset_index(drop=True)

# =========================
# TARGET CREATION (MOVE HERE)
# =========================

df['risk_score'] = (
    df['temperature_2m_max'].rank(pct=True) +
    df['wind_speed_10m_max'].rank(pct=True) +
    (1 - df['precipitation_sum'].rank(pct=True))
)

df['fire_risk_proxy'] = (df['risk_score'] > df['risk_score'].median()).astype(int)

# =========================
# VALIDATION (IMPORTANT SAFETY CHECK)
# =========================

print("Class distribution:")
print(df['fire_risk_proxy'].value_counts())

# Ensure valid classification
if df['fire_risk_proxy'].nunique() < 2:
    raise ValueError("fire_risk_proxy has only one class — cannot compute correlation.")

# =========================
# CLEAN DATA
# =========================

df_corr = df.dropna().copy()

# =========================
# CORRELATION MATRIX
# =========================

numeric_cols = df_corr.select_dtypes(include=['float64', 'float32', 'int64', 'int32'])
corr_matrix = numeric_cols.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# =========================
# HEATMAP VISUALIZATION
# =========================

plt.figure(figsize=(12, 8))

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 8}
)

plt.title("Correlation Matrix of Weather Variables", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# =========================
# FIRE RISK CORRELATIONS
# =========================

print("\n--- Correlations with Fire Risk Proxy ---")
print(
    corr_matrix['fire_risk_proxy']
    .sort_values(ascending=False)
    .round(3)
)

In [ ]:
# =========================
# 1. FEATURE ENGINEERING
# =========================

df['is_rain'] = (df['precipitation_sum'] > 1.0).astype(int)

df['month'] = pd.to_datetime(df['date']).dt.month



# IMPORTANT: sort BEFORE lag/rolling
df = df.sort_values(["location", "date"]).reset_index(drop=True)

# Lag feature
df['temp_lag1'] = df.groupby('location')['temperature_2m_max'].shift(1)

# Rolling averages (safe)
df['temp_3day_avg'] = df.groupby('location')['temperature_2m_max'] \
    .transform(lambda x: x.rolling(3, min_periods=1).mean())

df['wind_3day_avg'] = df.groupby('location')['wind_speed_10m_max'] \
    .transform(lambda x: x.rolling(3, min_periods=1).mean())


# =========================
# FIXED DAYS SINCE RAIN
# =========================

# Step 1: define rain event
df['rain_event'] = df['precipitation_sum'] > 1.0

# Step 2: create rain groups (each continuous rainy segment)
df['rain_group'] = df.groupby('location')['rain_event'].cumsum()

# Step 3: compute days since rain within each group
df['days_since_rain'] = df.groupby(['location', 'rain_group']).cumcount()

# Step 4: cleanup helper columns
df = df.drop(columns=['rain_event', 'rain_group'])


# =========================
# DROP NA AFTER FEATURES
# =========================

df = df.dropna().reset_index(drop=True)

# =========================
# 2. FEATURES & TARGET
# =========================

X_f = df[
    [
        'temperature_2m_max',
        'temp_lag1',
        'precipitation_sum',
        'days_since_rain',
        'temp_3day_avg',
        'wind_3day_avg',
        'month'
    ]
]

y_f = df['fire_risk_proxy']

# =========================
# TIME-BASED SPLIT (NO SHUFFLE, NO LEAKAGE)
# =========================

split_index = int(len(df) * 0.8)

X_train_f = X_f.iloc[:split_index]
X_test_f = X_f.iloc[split_index:]

y_train_f = y_f.iloc[:split_index]
y_test_f = y_f.iloc[split_index:]

# =========================
# 4. SMOTE OVERSAMPLING
# =========================

# NOTE:
# SMOTE is used here for imbalance handling,
# but it may slightly break temporal realism since it generates synthetic samples.
# This is acceptable for baseline experimentation.

smote = SMOTE(random_state=42)

X_train_f_resampled, y_train_f_resampled = smote.fit_resample(
    X_train_f, y_train_f
)

# =========================
# 5. MODELS
# =========================

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_f_resampled, y_train_f_resampled)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_f_resampled, y_train_f_resampled)

# =========================
# 6. PREDICTIONS
# =========================

rf_pred = rf_model.predict(X_test_f)
log_pred = log_model.predict(X_test_f)

# =========================
# 7. EVALUATION
# =========================

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "F1 Score": [
        f1_score(y_test_f, log_pred),
        f1_score(y_test_f, rf_pred)
    ],
    "Accuracy": [
        accuracy_score(y_test_f, log_pred),
        accuracy_score(y_test_f, rf_pred)
    ]
})

print(results)

print("\nClass distribution:")
print(y_test_f.value_counts(normalize=True))

# =========================
# 8. CONFUSION MATRIX (RF)
# =========================

cm = confusion_matrix(y_test_f, rf_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title("Random Forest Confusion Matrix")
plt.show()

# =========================
# 9. ROC CURVE (RF)
# =========================

y_prob = rf_model.predict_proba(X_test_f)[:, 1]

fpr, tpr, _ = roc_curve(y_test_f, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], '--')
plt.title("ROC Curve - Random Forest")
plt.legend()
plt.show()

In [ ]:
print(df['fire_risk_proxy'].value_counts(normalize=True))

## Target Variable Construction (Fire Risk Proxy)

Since real wildfire occurrence data is not available, a proxy target variable was constructed using a composite risk score derived from environmental conditions.

The risk score combines:
- Temperature (higher values increase risk)
- Wind speed (higher values increase fire spread risk)
- Precipitation (inverse relationship, lower rainfall increases risk)

This results in a continuous risk indicator which is then converted into a binary classification target:
- **1 = High fire risk**
- **0 = Low fire risk**

---

## Class Distribution

The dataset shows a near-balanced distribution:

- Class 0 (low risk): 5626 samples  
- Class 1 (high risk): 5624 samples  

This balance ensures that the classification model is not biased toward a dominant class and improves training stability.

---

## Correlation Analysis

A correlation matrix was computed to understand relationships between environmental variables and fire risk.

### Key Findings:

Strong positive correlations with fire risk:
- Risk score: **0.832**
- Temperature max: **0.674**
- Temperature mean: **0.647**
- Apparent temperature: **0.616**
- Wind speed: **0.503**

Moderate negative correlation:
- Precipitation: **-0.244**

Weak spatial influence:
- Longitude: **0.094**
- Latitude: **-0.116**

### Interpretation:

Fire risk is primarily driven by **heat and wind conditions**, while precipitation reduces risk. Geographic location has minimal influence compared to weather dynamics.

---

## Machine Learning Models

Two classification models were trained:

### Logistic Regression
- F1 Score: **0.8469**
- Accuracy: **0.8785**

### Random Forest
- F1 Score: **0.8947**
- Accuracy: **0.9159**

---

## Model Comparison

The Random Forest model outperforms Logistic Regression in both metrics.

This indicates:
- Strong non-linear relationships in the data
- Feature interactions between weather variables
- Better robustness to complex patterns

---

## Confusion Matrix Interpretation

The Random Forest model shows:
- High correct classification rate for both classes
- Balanced false positives and false negatives
- Strong generalization performance

---

## Final Class Distribution (After Processing)

After preprocessing and feature engineering:

- Class 0: ~50%
- Class 1: ~50%

This confirms balanced training data and stable evaluation conditions.

---

## ROC Curve & AUC

The AUC of 0.96 indicates a near-perfect ability of the model to distinguish between safe days and high-risk days. The steepness of the curve demonstrates that the model maintains a high True Positive rate even at very low False Positive thresholds.

---

## Overall Interpretation

Our results demonstrate that the relationship between atmospheric conditions and forest fire risk in Azerbaijan is statistically predictable:

- Higher temperature = higher fire risk
- Higher wind speed = higher fire risk
- Higher precipitation = lower fire risk

However, the current system is based on a **proxy label**, not real wildfire events.

---

## Future Improvements

- Improvements: Replace the proxy label with NASA FIRMS historical fire coordinates to move from a weather-based proxy to an empirical fire model.
- Future Extension: Incorporate vegetation "fuel load" maps (NDVI) and upgrade the backend to XGBoost to better handle complex, non-linear weather interactions.